# Delovni tok Human-in-the-Loop z Microsoft Agent Framework

## 🎯 Cilji učenja

V tej beležki se boste naučili, kako uvesti delovne tokove **human-in-the-loop** z uporabo `RequestInfoExecutor` v Microsoft Agent Framework. Ta močan vzorec vam omogoča, da ustavite AI delovne tokove za zbiranje človeških vnosov, s čimer postanejo vaši agenti interaktivni in ljudem dajejo nadzor nad ključnimi odločitvami.

## 🔄 Kaj je Human-in-the-Loop?

**Human-in-the-loop (HITL)** je načrtovalni vzorec, kjer AI agenti ustavijo izvajanje in zahtevajo človeški vnos, preden nadaljujejo. To je bistveno za:

- ✅ **Kritične odločitve** - pridobite človeško odobritev pred pomembnimi dejanji
- ✅ **Dvoumne situacije** - dovolite ljudem, da pojasnijo, kadar je AI negotov
- ✅ **Uporabniške preference** - vprašajte uporabnike, naj izberejo med več možnostmi
- ✅ **Skladnost in varnost** - zagotovite človeški nadzor za regulirane operacije
- ✅ **Interaktivne izkušnje** - ustvarite pogovorne agente, ki se odzivajo na uporabniški vnos

## 🏗️ Kako deluje v Microsoft Agent Framework

Okvir zagotavlja tri ključne komponente za HITL:

1. **`RequestInfoExecutor`** - poseben izvrševalnik, ki ustavi delovni tok in pošlje `RequestInfoEvent`
2. **`RequestInfoMessage`** - osnovna razred za tipizirane zahtevane obremenitve, poslane ljudem
3. **`RequestResponse`** - povezuje človeške odzive z izvirnimi zahtevami z uporabo `request_id`

**Vzorec delovnega toka:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 Naš primer: rezervacija hotela s potrditvijo uporabnika

Nadgradili bomo pogojni delovni tok z dodajanjem človeške potrditve **preden** predlagamo alternativne destinacije:

1. Uporabnik zahteva destinacijo (npr. "Pariz")
2. `availability_agent` preveri, ali so sobe na voljo
3. **Če ni sob** → `confirmation_agent` vpraša "Ali bi radi videli alternative?"
4. Delovni tok se **ustavi** z uporabo `RequestInfoExecutor`
5. **Človek odgovori** "da" ali "ne" prek vnosa v konzolo
6. `decision_manager` usmeri glede na odgovor:
   - **Da** → Pokaži alternativne destinacije
   - **Ne** → Prekliči zahtevo za rezervacijo
7. Prikaži končni rezultat

To prikazuje, kako omogočiti uporabnikom nadzor nad agentovimi predlogi!

---

Začnimo! 🚀


## Korak 1: Uvoz potrebnih knjižnic

Uvozimo standardne komponente ogrodja Agent plus **razrede specifične za človeka v zanki**:
- `RequestInfoExecutor` - Izvrševalec, ki zaustavi potek dela za človeški vnos
- `RequestInfoEvent` - Dogodek, ki se sproži, ko je zahtevan človeški vnos
- `RequestInfoMessage` - Osnovni razred za tipizirane zahtevane vsebine
- `RequestResponse` - Povezuje človeške odgovore z zahtevami
- `WorkflowOutputEvent` - Dogodek za zaznavanje izhodov poteka dela


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    Executor,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowRunState,          # Enum of workflow run states
    executor,
    handler,
    response_handler,          # NEW: decorator for the human-feedback response handler
    tool,
)

from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop uses ctx.request_info() + @response_handler")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## Korak 2: Določitev Pydantic modelov za strukturirane izhode

Ti modeli določajo **shemo**, ki jo bodo agenti vračali. Ohranjamo vse modele iz pogojnega poteka in dodamo:

**Novo za človeka-v-zanki:**
- `HumanFeedbackRequest` - Podrazred `RequestInfoMessage`, ki določa zahtevo, poslano ljudem
  - Vsebuje `prompt` (vprašanje za zastavljanje) in `destination` (kontekst o nedosegljivem mestu)


In [ ]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Plain dataclass payload for ctx.request_info()
@dataclass
class HumanFeedbackRequest:
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## Korak 3: Ustvarite orodje za rezervacijo hotela

Enako orodje kot v pogojnem delovnem toku - preverja, ali so sobe na voljo na destinaciji.


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

✅ hotel_booking tool created with @ai_function decorator


## Korak 4: Določitev funkcij pogojev za usmerjanje

Potrebujemo **štiri funkcije pogojev** za naš delovni tok s človekom v zanki:

**Iz pogojnega delovnega toka:**
1. `has_availability_condition` - Usmerja, ko so hoteli na voljo
2. `no_availability_condition` - Usmerja, ko hoteli niso na voljo

**Novo za človeka v zanki:**
3. `user_wants_alternatives_condition` - Usmerja, ko uporabnik reče "da" alternativam
4. `user_declines_alternatives_condition` - Usmerja, ko uporabnik reče "ne" alternativam


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## Korak 5: Ustvarite izvajalca Decision Manager

To je **jedro vzorca človek-v-zanki**! `DecisionManager` je prilagojen `Executor`, ki:

1. **Prejema povratne informacije od ljudi** prek objektov `RequestResponse`
2. **Obdeluje uporabnikovo odločitev** (da/ne)
3. **Usmerja potek dela** s pošiljanjem sporočil ustreznim agentom

Ključne lastnosti:
- Uporablja dekorator `@handler` za izpostavitev metod kot korakov poteka dela
- Prejema `RequestResponse[HumanFeedbackRequest, str]`, ki vsebuje tako izvorno zahtevo kot uporabnikov odgovor
- Izda preprosta sporočila „da“ ali „ne“, ki sprožijo naše funkcije pogojev


In [ ]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_confirmation(
        self,
        response: AgentExecutorResponse,
        ctx: WorkflowContext,
    ) -> None:
        """Parse the confirmation question and pause the workflow for human input."""
        confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                <strong>🔄 Requesting human input:</strong> {confirmation.question}
            </div>
        """)
        )
        # Pause the workflow; the human reply (a string) is delivered to on_human_feedback below.
        await ctx.request_info(
            request_data=HumanFeedbackRequest(
                prompt=confirmation.question,
                destination=confirmation.destination,
            ),
            response_type=str,
        )

    @response_handler
    async def on_human_feedback(
        self,
        original_request: HumanFeedbackRequest,
        feedback: str,
        ctx: WorkflowContext[AgentExecutorRequest, str],
    ) -> None:
        """Route the workflow based on the human's yes/no reply."""
        user_reply = (feedback or "").strip().lower()
        destination = original_request.destination or "unknown"

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = Message(
                role="user",
                contents=[f"The user wants to see alternative destinations near {destination}. Please suggest one."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → cancellation_agent
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = Message(
                role="user",
                contents=["The user has declined to see alternatives. Please acknowledge their decision."],
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created (@handler pauses via request_info, @response_handler routes)")

✅ DecisionManager executor created with @handler method for human feedback


## Korak 6: Ustvarite po meri prilagojenega izvrševalca prikaza

Enak izvrševalec prikaza kot pri pogojnem delovnem toku - vrne končne rezultate kot izhod delovnega toka.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## Korak 7: Naloži okoljske spremenljivke

Konfigurirajte LLM odjemalca (Microsoft Foundry / Azure OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

# Microsoft Foundry provider with keyless AzureCliCredential auth (run `az login`).
# Matches the pattern used across lessons 01-13 and the other Lesson 14 notebooks.
chat_client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)

print("✅ Chat client configured with Microsoft Foundry")


✅ Chat client configured with GitHub Models


## Korak 8: Ustvarjanje AI agentov in izvajalcev

Ustvarimo **šest komponent poteka dela**:

**Agenti (oviti v AgentExecutor):**
1. **availability_agent** - Preverja razpoložljivost hotela z uporabo orodja
2. **confirmation_agent** - 🆕 Pripravlja zahtevo za človeško potrditev
3. **alternative_agent** - Predlaga alternativna mesta (ko uporabnik reče da)
4. **booking_agent** - Spodbuja rezervacijo (ko so sobe na voljo)
5. **cancellation_agent** - 🆕 Upravlja sporočilo o preklicu (ko uporabnik reče ne)

**Posebni izvajalci:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor`, ki zaustavi potek dela za človeški vnos
7. **decision_manager** - 🆕 Po meri narejen izvajalec, ki usmerja glede na človeški odgovor (že definiran zgoraj)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        default_options={"response_format": ConfirmationQuestion},  # Structured JSON output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        default_options={"response_format": CancellationMessage},
    ),
    id="cancellation_agent",
)

# DecisionManager instance - pauses for human input (request_info) and routes on the reply
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## Korak 9: Izgradnja poteka dela s človekom v zanki

Zdaj zgradimo graf poteka dela s **pogojnim usmerjanjem**, ki vključuje pot z človekom v zanki:

**Struktura poteka dela:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**Ključne povezave:**
- `availability_agent → confirmation_agent` (ko ni sob)
- `confirmation_agent → prepare_human_request` (pretvorba vrste)
- `prepare_human_request → request_info_executor` (premor za človeka)
- `request_info_executor → decision_manager` (vedno - zagotavlja RequestResponse)
- `decision_manager → alternative_agent` (ko uporabnik reče "da")
- `decision_manager → cancellation_agent` (ko uporabnik reče "ne")
- `availability_agent → booking_agent` (ko so sobe na voljo)
- Vse poti se končajo pri `display_result`


In [ ]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, decision_manager)  # decision_manager pauses via ctx.request_info
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## Korak 10: Zaženi testni primer 1 - Mesto BREZ razpoložljivosti (Pariz s človeškim potrjevanjem)

Ta test prikazuje **popoln cikel s človekom v zanki**:

1. Zahteva za hotel v Parizu
2. availability_agent preveri → Ni sob
3. confirmation_agent ustvari vprašanje za človeka
4. request_info_executor **zaustavi potek dela** in sproži `RequestInfoEvent`
5. **Aplikacija zazna dogodek in pozove uporabnika v konzoli**
6. Uporabnik vnese "da" ali "ne"
7. Aplikacija pošlje odgovor preko `send_responses_streaming()`
8. decision_manager usmeri na podlagi odgovora
9. Prikazan je končni rezultat

**Ključni vzorec:**
- Uporabi `workflow.run_stream()` za prvo iteracijo
- Uporabi `workflow.send_responses_streaming(pending_responses)` za nadaljnje iteracije
- Poslušaj `RequestInfoEvent` za zaznavanje, kdaj je potreben človeški vnos
- Poslušaj `WorkflowOutputEvent` za zajem končnih rezultatov


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], 
    should_respond=True
)

# Human-in-the-loop execution pattern.
# We script the human's answer ("yes") instead of input() so the notebook runs unattended.
# In a real app, replace SCRIPTED_ANSWER with input() or a UI callback.
SCRIPTED_ANSWER = "yes"
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

# First run streams events; resume with run(stream=True, responses=...) after each pause.
stream = workflow.run(request_paris, stream=True)
while True:
    requests: list[tuple[str, HumanFeedbackRequest]] = []
    async for event in stream:
        if event.type == "request_info" and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            print(f"   Question: {event.data.prompt}")
            requests.append((event.request_id, event.data))
        elif event.type == "output":
            workflow_output = str(event.data)
            print(f"\n✅ Workflow completed with output!")

    if not requests:
        break

    # Provide the (scripted) human answer for each pending request.
    responses: dict[str, str] = {}
    for req_id, req in requests:
        answer = SCRIPTED_ANSWER
        print(f"\n📝 (scripted) You answered: {answer}")
        responses[req_id] = answer

    stream = workflow.run(stream=True, responses=responses)

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## Korak 11: Zaženi testni primer 2 - Mesto Z Dovoljjo Prosti Kapacitetami (Stockholm - brez človeškega posredovanja)

Ta test prikazuje **direktno pot**, ko so sobe na voljo:

1. Zahteva hotel v Stockholmu
2. availability_agent preveri → Sobe so na voljo ✅
3. booking_agent predlaga rezervacijo
4. display_result prikazuje potrditev
5. **Brez potrebe po človeški interakciji!**

Delovni postopek popolnoma zaobide pot z vključenostjo človeka, kadar so sobe na voljo.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## Ključni poudarki in najboljše prakse human-in-the-loop

### ✅ Kaj ste se naučili:

#### 1. **Vzorec RequestInfoExecutor**
Vzorec human-in-the-loop v Microsoft Agent Frameworku uporablja tri ključne komponente:
- `RequestInfoExecutor` - Zaustavi potek in sproži dogodke
- `RequestInfoMessage` - Osnovna razred za tipizirane zahteve (naredite podrazred!)
- `RequestResponse` - Povezuje odzive ljudi z izvirnimi zahtevami

**Ključno razumevanje:**
- `RequestInfoExecutor` sam ne zbira vnosa - samo začasno ustavi potek
- Vaša aplikacija mora poslušati `RequestInfoEvent` in zbrati vnos
- Poklicati morate `send_responses_streaming()` s slovarjem, ki preslika `request_id` na odgovor uporabnika

#### 2. **Vzorec izvajanja z pretakanjem (streaming)**
```python
# Prva ponovitev
stream = workflow.run_stream(initial_request)

# Naslednje ponovitve (po človeškem vnosu)
stream = workflow.send_responses_streaming(pending_responses)

# Dogodke vedno obdeluj
events = [event async for event in stream]
```

#### 3. **Dogodkovno usmerjena arhitektura**
Poslušajte določene dogodke za nadzor poteka:
- `RequestInfoEvent` - Potreben je človeški vnos (potek začasno ustavljen)
- `WorkflowOutputEvent` - Končni rezultat je na voljo (potek zaključen)
- `WorkflowStatusEvent` - Spremembe stanja (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS itd.)

#### 4. **Lastni executorji s @handler**
`DecisionManager` prikazuje, kako ustvariti executorje, ki:
- Uporabljajo dekorator `@handler`, da razkrijejo metode kot korake poteka
- Prejemajo tipizirana sporočila (npr. `RequestResponse[HumanFeedbackRequest, str]`)
- Usmerjajo potek s pošiljanjem sporočil drugim executorjem
- Dostopajo do konteksta preko `WorkflowContext`

#### 5. **Pogojno usmerjanje z odločbami ljudi**
Lahko ustvarite pogoje funkcij, ki ocenjujejo človeške odzive:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 Resnični primeri uporabe:

1. **Poteki odobritve**
   - Pridobite odobritev nadrejenega pred obdelavo stroškovnih poročil
   - Zahtevajte človeški pregled pred pošiljanjem avtomatiziranih e-pošt
   - Potrdite transakcije z visokimi vrednostmi pred izvedbo

2. **Moderiranje vsebin**
   - Označite vprašljive vsebine za človeški pregled
   - Prosite moderatorje, naj sprejmejo končno odločitev v obrobnih primerih
   - Razveljavite na ljudi, kadar je zaupanje AI nizko

3. **Poprodajne storitve**
   - Naj AI samodejno obravnava rutinska vprašanja
   - Zapletene zadeve prenesite človeškim agentom
   - Vprašajte stranko, če želi govoriti s človekom

4. **Obdelava podatkov**
   - Prosite ljudi, naj rešijo nejasne vnose podatkov
   - Potrdite AI interpretacije nejasnih dokumentov
   - Pustite uporabnikom izbiro med več veljavnimi interpretacijami

5. **Varnostno kritični sistemi**
   - Zahtevajte človeško potrditev pred nepovratnimi dejanji
   - Pridobite odobritev pred dostopom do občutljivih podatkov
   - Potrdite odločitve v reguliranih panogah (zdravstvo, finance)

6. **Interaktivni agenti**
   - Zgradite pogovorne bote, ki zastavljajo nadaljnja vprašanja
   - Ustvarite čarovnike, ki vodijo uporabnike skozi zahtevne postopke
   - Načrtujte agente, ki sodelujejo s človekom korak za korakom

### 🔄 Primerjava: S human-in-the-loop ali brez

| Funkcija | Pogojni potek | Potek z human-in-the-loop |
|---------|---------------------|---------------------------|
| **Izvajanje** | Enkratni `workflow.run()` | Zanka z `run_stream()` + `send_responses_streaming()` |
| **Uporabniški vnos** | Ni vnosa (popolnoma avtomatizirano) | Interaktivna pozivna okna preko `input()` ali UI |
| **Komponente** | Agenti + Executorji | + RequestInfoExecutor + DecisionManager |
| **Dogodki** | Samo AgentExecutorResponse | RequestInfoEvent, WorkflowOutputEvent itd. |
| **Zaustavljanje** | Brez zaustavljanja | Potek se ustavi pri RequestInfoExecutor |
| **Človeški nadzor** | Brez človeškega nadzora | Ljudje sprejemajo ključne odločitve |
| **Uporabniški primer** | Avtomatizacija | Sodelovanje & nadzor |

### 🚀 Napredni vzorci:

#### Več človeških odločilnih točk
V istem poteku lahko imate več `RequestInfoExecutor` vozlišč:
```python
.add_edge(agent1, request_info_1)  # Prva človeška odločitev
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # Druga človeška odločitev
.add_edge(decision_manager_2, final_agent)
```

#### Upravljanje s časovnimi omejitvami
Implementirajte časovne omejitve za človeške odzive:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # Privzeto na varno možnost
```

#### Obsežna integracija UI
Namesto `input()`, integrirajte z spletnim UI, Slack, Teams itd.:
```python
if isinstance(event, RequestInfoEvent):
    # Pošlji obvestilo na uporabnikov priljubljen kanal
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### Pogojni human-in-the-loop
Človeškega vnosa zahtevajte samo v določenih situacijah:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # Usmeri le k človeku, če je zaupanje nizko ali vrednost visoka
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ Najboljše prakse:

1. **Vedno naredite podrazred RequestInfoMessage**
   - Zagotavlja varnost tipov in validacijo
   - Omogoča bogat kontekst za upodabljanje UI
   - Poenostavi namen vsake vrste zahteve

2. **Uporabljajte opisne pozive**
   - Vključite kontekst o tem, kar sprašujete
   - Pojasnite posledice posamezne izbire
   - Ohranite vprašanja enostavna in jasna

3. **Obdelujte nepričakovane vnose**
   - Validirajte uporabniške odgovore
   - Ponudite privzete vrednosti za neveljaven vnos
   - Zagotovite jasna sporočila o napaki

4. **Spremljajte ID-je zahtev**
   - Uporabite korelacijo med request_id in odgovori
   - Ne upravljajte stanja ročno

5. **Načrtujte za neblokiranje**
   - Ne blokirajte niti med čakanjem na vnos
   - Uporabljajte asinhrone vzorce povsod
   - Podpirajte sočasne primere potekov

### 📚 Sorodni koncepti:

- **Agent Middleware** - Prestrezanje klicev agentov (prejšnji zvezek)
- **Upravljanje stanja poteka** - Ohranjanje stanja poteka med izvajanji
- **Sodelovanje več agentov** - Združevanje human-in-the-loop z agentnimi ekipami
- **Dogodkovno usmerjene arhitekture** - Gradnja reaktivnih sistemov z dogodki

---

### 🎓 Čestitke!

Obvladali ste human-in-the-loop poteke z Microsoft Agent Frameworkom! Zdaj veste, kako:
- ✅ Začasno ustaviti potek za zbiranje človeškega vnosa
- ✅ Uporabiti RequestInfoExecutor in RequestInfoMessage
- ✅ Obdelovati izvajanje s pretakanjem in dogodki
- ✅ Ustvariti lastne executorje s @handler
- ✅ Usmerjati poteke na podlagi človeških odločitev
- ✅ Zgraditi interaktivne AI agente, ki sodelujejo z ljudmi

**To je ključni vzorec za gradnjo zaupanja vrednih, nadzorovanih AI sistemov!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Omejitev odgovornosti**:
Ta dokument je bil preveden z uporabo AI prevajalske storitve [Co-op Translator](https://github.com/Azure/co-op-translator). Čeprav si prizadevamo za natančnost, vas prosimo, da upoštevate, da avtomatizirani prevodi lahko vsebujejo napake ali netočnosti. Izvirni dokument v njegovem izvirnem jeziku je treba obravnavati kot avtoritativni vir. Za kritične informacije je priporočljiv strokovni človeški prevod. Ne odgovarjamo za morebitna nesporazume ali napačne interpretacije, ki izhajajo iz uporabe tega prevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
